# 📊 Exploratory Data Analysis — Customer Churn

This notebook explores the Telco Customer Churn dataset to understand:
- Overall churn distribution
- Churn by demographics and service type
- Correlation of numerical features with churn
- Key patterns that inform our model strategy

In [ ]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
from data_loader import load_config, download_data, load_raw_data, clean_data
plt.rcParams["figure.figsize"] = (12, 5)
sns.set_palette("Set2")
print("✅ Setup complete")

In [ ]:
config = load_config("../config.yaml")
download_data(config["data"]["download_url"], "../" + config["data"]["raw_path"])
df_raw = load_raw_data("../" + config["data"]["raw_path"])
df = clean_data(df_raw, config)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Churn Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
churn_counts = df["Churn"].value_counts()
labels = ["No Churn", "Churn"]
colors = ["#42A5F5", "#EF5350"]
axes[0].pie(churn_counts.values, labels=labels, colors=colors, autopct="%1.1f%%", startangle=140, wedgeprops={"edgecolor":"white","linewidth":2})
axes[0].set_title("Overall Churn Distribution", fontsize=14, fontweight="bold")
axes[1].bar(labels, churn_counts.values, color=colors, edgecolor="white", width=0.5)
for i, v in enumerate(churn_counts.values):
    axes[1].text(i, v+30, str(v), ha="center", fontweight="bold")
axes[1].set_title("Churn Count", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Number of Customers")
plt.suptitle("Customer Churn Overview", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Churn by Key Categorical Features
cat_features = ["Contract", "InternetService", "PaymentMethod", "TechSupport"]
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, feature in zip(axes.flatten(), cat_features):
    churn_rate = df.groupby(feature)["Churn"].mean() * 100
    churn_rate = churn_rate.sort_values(ascending=False)
    bars = ax.bar(churn_rate.index, churn_rate.values, color=plt.cm.RdYlGn_r(churn_rate.values/100), edgecolor="white")
    for bar, val in zip(bars, churn_rate.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f"{val:.1f}%", ha="center", va="bottom")
    ax.set_title(f"Churn Rate by {feature}", fontsize=12, fontweight="bold")
    ax.set_ylabel("Churn Rate (%)")
    ax.tick_params(axis="x", rotation=15)
    ax.grid(axis="y", alpha=0.3)
plt.suptitle("Churn Rate by Key Features", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Numerical Feature Distributions
num_features = ["tenure", "MonthlyCharges", "TotalCharges"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, num_features):
    for label, color, name in [(0, "#42A5F5", "No Churn"), (1, "#EF5350", "Churn")]:
        df[df["Churn"] == label][col].plot.kde(ax=ax, color=color, lw=2.5, label=name)
    ax.set_title(f"{col} by Churn", fontsize=12, fontweight="bold")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Tenure vs Monthly Charges scatter
plt.figure(figsize=(10,6))
scatter = plt.scatter(df["tenure"], df["MonthlyCharges"], c=df["Churn"], cmap="RdYlGn_r", alpha=0.4, s=15)
plt.colorbar(scatter, label="Churn (1=Yes)")
plt.xlabel("Tenure (months)")
plt.ylabel("Monthly Charges ($)")
plt.title("Tenure vs Monthly Charges — coloured by Churn", fontsize=13, fontweight="bold")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("📌 High-charge, low-tenure customers churn the most")